# B0-P2 — Balanced Data & Extended Training Pilot
Tesla T4, Internet and existing GITHUB_TOKEN. One Save & Run All: CPU gates → verified prior timings → temporary GPU benchmarks → common1024/768/640 budget freeze → three fresh arms → complete independent downloads/hashes → checkpoint re-evaluation → paired Episode-cluster analysis and return. Cumulative GPU-phase cap7200 seconds including all benchmarks, diagnostic/evaluation forwards, checkpoint and restore; planning cap6480. No Test,12M, main three-seed experiment or M6. Preserve and attach complete Output for exact-source resume within the remaining original budget.

In [ ]:
from pathlib import Path
import json,os,signal,subprocess,sys,time
started=time.monotonic()
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['OMP_NUM_THREADS']='1'
os.environ['PYTEST_DISABLE_PLUGIN_AUTOLOAD']='1'
repo=Path('/kaggle/working/minicells-b0-p2')
branch='research/epclm-004-b0-p2'
def run(args,limit=300,log=None):
    remaining=min(limit,started+10500-time.monotonic())
    if remaining<=0:raise TimeoutError('End-to-end session limit; preserve complete Output')
    stream=open(log,'a',buffering=1) if log else None
    try:
        process=subprocess.Popen(args,cwd=repo if repo.exists() else None,start_new_session=True,
            stdout=stream,stderr=subprocess.STDOUT if stream else None)
        try:code=process.wait(timeout=remaining)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGTERM)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()
            raise TimeoutError('Session stopped; P2 Output retained; do not change the frozen budget')
    finally:
        if stream:stream.close()
    if log:print(Path(log).read_text(),flush=True)
    if code:raise RuntimeError('P2 stopped; inspect execution-error.json and logs in Output')
if not repo.exists():run(['git','clone','--depth=1','--branch',branch,'https://github.com/ArcheLabs/mini-cells.git',str(repo)])
else:
    origin=subprocess.check_output(['git','remote','get-url','origin'],cwd=repo,text=True).strip()
    if origin.removesuffix('.git')!='https://github.com/ArcheLabs/mini-cells':raise ValueError('Unrelated checkout')
    run(['git','fetch','--depth=1','origin',branch]);run(['git','switch','--detach','FETCH_HEAD'])
base=repo/'research/projects/cellular-language-models/experiments/epclm-004-unified-memory-10m/m5-sv/b0-p2'
run([sys.executable,'-m','pip','install','-q','--no-deps','einops==0.8.2'])
source=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
out=Path('/kaggle/working/b0-p2-'+source[:12]);out.mkdir(exist_ok=True)
proof=out/'p2-cpu-preflight.json'
if not proof.exists() and Path('/kaggle/input').exists():
    for prior in Path('/kaggle/input').rglob('p2-cpu-preflight.json'):
        if json.loads(prior.read_text())=={'source_commit':source,'status':'PASS'}:
            proof.write_text(prior.read_text())
            if (prior.parent/'cpu-tests.log').exists():(out/'cpu-tests.log').write_text((prior.parent/'cpu-tests.log').read_text())
            break
if not proof.exists():
    run([sys.executable,'-m','pytest','-q',str(base/'tests')],limit=300,log=out/'cpu-tests.log')
    proof.write_text(json.dumps({'source_commit':source,'status':'PASS'}))
#One automatic sequence. Formal P2 updates cannot start before measured common-budget freeze.
run([sys.executable,str(base/'p2_run.py'),'--output',str(out),'--execute'],limit=10200,log=out/'raw.log')
(out/'notebook-completion.json').write_text(json.dumps({'source_commit':source,'end_to_end_wall_seconds':time.monotonic()-started,'scope':'Notebook setup, CPU gates, controller, publication and independent verification'}))
